# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count
import pandas as pd

spark = SparkSession.builder.appName("PatentJoin").getOrCreate()

# Joining citations with cited patent state

In [9]:
patents_cited = patents.select("PATENT", "POSTATE").alias("cited_patents")

cited_df = citations.join(
    patents_cited,
    citations["CITED"] == col("cited_patents.PATENT"),
    "left"
).withColumnRenamed("POSTATE", "CITED_STATE") \
 .drop("PATENT")  


# Join to get citing patent state

In [10]:
patents_citing = patents.select("PATENT", "POSTATE").alias("citing_patents")

citing_df = cited_df.join(
    patents_citing,
    cited_df["CITING"] == col("citing_patents.PATENT"),
    "left"
).withColumnRenamed("POSTATE", "CITING_STATE") \
 .drop("PATENT")  

# Computing self state citation flag

In [11]:
citing_df = citing_df.withColumn(
    "self_cited",
    when(col("CITED_STATE") == col("CITING_STATE"), 1).otherwise(0)
)


# Aggregating self-state counts per citing patent


In [12]:
self_state_counts = citing_df.groupBy("CITING") \
                             .agg(count(when(col("self_cited") == 1, True)).alias("self_state_count"))

# Joining back with original patent data to augment

In [13]:
patents_augmented = patents.join(
    self_state_counts,
    patents["PATENT"] == self_state_counts["CITING"],
    "left"
).drop("CITING")


# Top 10 patents by self state citations

In [14]:
top10_full = patents_augmented.orderBy(col("self_state_count").desc(), col("PATENT")).limit(10)


# Displaying in the notebook

In [15]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200) 
pd.set_option('display.max_colwidth', None)

top10_pd = top10_full.toPandas()
top10_pd

,PATENT,GYEAR,GDATE,APPYEAR,COUNTRY,POSTATE,ASSIGNEE,ASSCODE,CLAIMS,NCLASS,CAT,SUBCAT,CMADE,CRECEIVE,RATIOCIT,GENERAL,ORIGINAL,FWDAPLAG,BCKGTLAG,SELFCTUB,SELFCTLB,SECDUPBD,SECDLWBD,self_state_count
0,5959466,1999,14515,1997,US,CA,5310.0,2,NaN,326,4,46,159,0,1.000,NaN,0.6186,NaN,4.8868,0.0455,0.0440,NaN,NaN,125
1,5983822,1999,14564,1998,US,TX,569900.0,2,NaN,114,5,55,200,0,0.995,NaN,0.7201,NaN,12.4500,0.0000,0.0000,NaN,NaN,103
2,6008204,1999,14606,1998,US,CA,749584.0,2,NaN,514,3,31,121,0,1.000,NaN,0.7415,NaN,5.0000,0.0085,0.0083,NaN,NaN,100
3,5952345,1999,14501,1997,US,CA,749584.0,2,NaN,514,3,31,118,0,1.000,NaN,0.7442,NaN,5.1102,0.0000,0.0000,NaN,NaN,98
4,5958954,1999,14515,1997,US,CA,749584.0,2,NaN,514,3,31,116,0,1.000,NaN,0.7397,NaN,5.1810,0.0000,0.0000,NaN,NaN,96
5,5998655,1999,14585,1998,US,CA,NaN,1,NaN,560,1,14,114,0,1.000,NaN,0.7387,NaN,5.1667,NaN,NaN,NaN,NaN,96
6,5936426,1999,14466,1997,US,CA,5310.0,2,NaN,326,4,46,178,0,1.000,NaN,0.5800,NaN,11.2303,0.0765,0.0730,NaN,NaN,94
7,5739256,1998,13983,1995,US,CA,70060.0,2,15.0,528,1,15,453,0,1.000,NaN,0.8232,NaN,15.1104,0.1124,0.1082,NaN,NaN,90
8,5913855,1999,14417,1997,US,CA,733846.0,2,NaN,606,3,32,242,0,1.000,NaN,0.7403,NaN,8.3595,0.0000,0.0000,NaN,NaN,90
9,5925042,1999,14445,1997,US,CA,733846.0,2,NaN,606,3,32,242,0,1.000,NaN,0.7382,NaN,8.3471,0.0000,0.0000,NaN,NaN,90
